# **Parte A: Regresión**

En esta notebook veremos cómo se entrena un modelo de **regresión lineal** utilizando un algoritmo de optimización iterativo denominado **descenso de gradiente (gradient descent)**, el cual permite encontrar un mínimo global (en el mejor de los casos) o local, de la función de costo o de pérdida *diferenciable*.

La idea de dicho algoritmo es dar pasos en *dirección opuesta* al gradiente de la función en un punto, ya que es la dirección de descenso más empinado (mayor pendiente).

Trabajaremos con datasets sintéticos y modelos simples (regresores lineales y logísticos).

***
Lo primero que haremos será importar las librerías necesarias para nuestra tarea: Matplotlib, Numpy, TensorFlow, entre otras.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

***
## Regresor Lineal

Un **regresor lineal** es un modelo lineal que establece la relación entre la variable dependiente ($\hat{y}$) y una o más variables independientes ($x$).

Un modelo de regresión lineal `multi-variado` puede expresarse como:

$$\hat{y}=x_0w_0+x_1w_1+...+x_iw_i+b$$

Durante el proceso de aprendizaje, el regresor calcula su salida en función de las entradas ($x_{i}$) (**features**) y sus **parámetros** ($w_{i}$ y $b$).

Dicha salida se compara con el valor **target** ($y$) y se obtiene un valor de error para cada muestra de nuestro dataset.

Para el caso `univariado` el modelo se simplifica y queda como:

$$\hat{y}=xw+b$$

***
### **Ejercicio A-1:**

Antes de entrenar un modelo de regresión lineal, debemos generar nuestro dataset o conjunto de datos, de manera aleatoria. 

Emplearemos la siguiente ecuación:

$$y=3*x+(rand-0.5)$$

In [ ]:
#Función que genera un dataset sintético
def gen_random_data(a):
    x = np.linspace(-1, 1, 100) #ndarray 1D (100,)
    error = (np.random.rand(*x.shape) - 0.5 )  #random.rand genera datos aleatorios entre 0 y 1
    y = a * x + error
    return x.astype(np.float32), y.astype(np.float32)  #convierto a float32

#OBS: Al usar TensorFlow, el modelo espera datos de tipo float32 por eso convertimos float64 a float32

x, y = gen_random_data(3)

plt.title('Dataset sintético')
plt.plot(x, y, 'ro')
plt.xlabel('x')
plt.ylabel('y')

print(type(x), x.dtype)

***
## Entrenamiento 

El objetivo del entrenamiento es encontrar los parámetros $w_{i}$ y $b$ del modelo que minimizen una `función de pérdida` $L(y, \hat{y})$:

Para el caso de **regresión lineal univariada** (un sólo feature), buscaremos:

$$\underset{w,b}{arg\,min}=L(y,xw+b)$$

## Función de pérdida

En este tipo de tarea se emplea el **error cuadrático medio** (`Mean Squared Error, MSE`) como función de pérdida:

$$MSE(y,\hat{y})=\frac{1}{N}\sum(y-\hat{y})^{2}$$

A continuación, definiremos el regresor lineal (función lineal) y la función de pérdida (MSE):

In [ ]:
#Definimos el regresor lineal
def lineal(x, w, b):
  y_hat = x @ w +b  #producto punto o escalar, equivale a hacer np.dot(x, w) + b 
  return y_hat

#Calculamos el error cuadrático medio (MSE)
def mse(y_true, y_pred):
  se = tf.pow(y_true - y_pred, 2)
  return tf.reduce_mean(se)  #reducir a la media equivale calcular el promedio

Para trabajar con TensorFlow debo convertir los ndarray de NumPy a Tensor

In [ ]:
#Convertimos de ndarray a tensores
x_tf = tf.constant(x[:, np.newaxis])
y_tf = tf.constant(y[:, np.newaxis])

print(type(x), x.shape)
print(type(x_tf), x_tf.shape)


Asignamos valores a $w$ y a $b$ para calcular el MSE a partir de los datos *y_tf* e *y_tf* de nuestro dataset sintético:

In [ ]:
#Inicializamos nuestros parámetros
w = tf.constant(3, shape=(1, 1),  dtype=tf.float32)  #matriz 1x1
b = tf.constant(0, dtype=tf.float32)  #escalar

#Calculamos las salidas de nuestro modelo para cada muestra de nuestro dataset
y_hat = lineal(x_tf, w, b)

print(y_hat.shape)
print(f'tipo: {type(y_hat)}')

#Calculamos la pérdida para el conjunto de datos
loss = mse(y_tf, y_hat)

print(f'loss value: {loss}')
print(f'loss shape: {loss.shape}')
print(f'loos type: {type(loss)}')

print(f'resultado del entrenamiento -> w:{w}. b:{b}, mse:{loss}')

#Graficamos la recta de ajuste a nuestra nube de puntos
plt.title(f'Dataset sintético y recta de ajuste. MSE = {loss}')
plt.plot(x, y, 'ro')
plt.plot(x,y_hat,'b')
plt.xlabel('x')
plt.ylabel('y')

Hata ahora fijamos el valor de $w$ y de $b$ y calculamos el error ($MSE$).
Dichos parámetros seguramente no son los que producen un menor MSE.
A continuación, buscaremos una forma de encontrar los mejores valores de dichos parámetros.

***

### **Ejercicio A-2:** 

Para encontrar los parámetros que minimizan la función de pérdida, podríamos generar diferentes valores de nuestros parámetros y ver cuales son los que producen el menor MSE. Por simplicidad solo exploraremos $w$ y fijaremos $b=0$.

In [ ]:
def exp_loss(y, x, ws):
    def single_loss(w):
        w = tf.constant(w, shape=(1, 1), dtype=tf.float32)        
        y_hat = lineal(x, w, 0)
        error = mse(y,y_hat).numpy()
        return error
    
    # Convertimos la función single_loss (que recibe escalares) 
    # a una función vectorizada que nos permite trabajar con  arrays
    s = np.vectorize(single_loss) 
    return s(ws)

ws = np.linspace(1, 5, 51)
loss = exp_loss(y_tf, x_tf, ws)

plt.title('Función de pérdida del regresor lineal univariado, b=0')
plt.plot(ws,loss)
plt.xlabel('w')
plt.ylabel('Loss')


***
## Gradient Descent

Ahora analicemos cómo encontrar el mejor parámetro *w*, el cual minimizará la función de pérdida. Para ello debemos calcular la derivada de la función de pérdida (*loss*) y  aplicar el algoritmo de optimización *descenso de gradiente*.

#### **Pendiente de la pérdida**

* Dado que nuestra función de pérdida tiene un solo mínimo, se podrían tomar 2 valores cercanos de manera de conocer en qué dirección es conveniente explorar. 

* El modelo lineal es una función que depende no solo de los datos de entrada ($x$), sino también del parámetro a aprender ($w$), entonces la notaremos como $h(x,w)$. 

* Para conocer la pendiente de la función de pérdida dado un parámetro (w en este caso) podemos hacer:

$$pendiente_w(w_{1}, w_{0})=\frac{MSE(y,h(x,w_{1}))-MSE(y,h(x,w_{0}))}{w_{1}-w_{0}}$$

* Se podría inicializar $w$ de forma aleatoria e ir actualizando su valor **en contra de la pendiente**, ya que la pendiente indica la dirección en la que crece la pérdida con respecto a $w$. 

* El problema es determinar `CUANTO` moverse. La opción más básica es moverse proporcionalmente a la pendiente. 

* Entonces para actualizar el valor del parámetro *w* tendremos que utilizar un hiperparámetro (lr: learning rate) y calcular:

$$w_{n+1}=w_n – lr*pendiente_w $$


* El problema de utilizar la pendiente es que nos agrega muchos cálculos y el hiperparámetro $\Delta$. Tomemos el límite con $\Delta\rightarrow 0$.

#### **Pendiente de $w$:**

* Desde el punto de vista de la pendiente de $w$, si definimos a $w_{1}=w_{0}+\Delta$ entonces:

$$\lim_{\Delta \rightarrow 0} pendiente_w(w_{0}+\Delta,w_{0})= \lim_{\Delta \rightarrow 0} \frac{MSE(y,h(x,w_{0}+\Delta, b))-MSE(y,h(x,w_{0}, b))}{\Delta} =\frac{dMSE(y,h(x,w,b))}{dw}$$

* Esta derivada se puede resolver aplicando la **regla de la cadena**:

$$\frac{dMSE(y,h(x,w,b))}{dw}=\frac{dMSE(y,h(x,w,b))}{d(h(x,w,b))}.\frac{d(h(x,w,b))}{dw}$$

* La primera derivada se resuelve también por regla de la cadena:

$$\frac{dMSE(y,h(x,w,b))}{d(h(x,w,b)}=\frac{d(\frac{1}{N}\sum(y-h(x,w,b))^{2}}{d(h(x,w,b))}=-\frac{2}{N}\sum(y-h(x,w,b))$$

* La segunda derivada se resuelve así:

$$\frac{dh(x,w,b)}{dw}=\frac{d(xw+b)}{dw}=x$$

* Finalmente, resulta que:

$$\frac{dMSE(y,h(x,w,b))}{dw}=\frac{dMSE(y,h(x,w,b))}{d(h(x,w,b))}.\frac{d(h(x,w,b))}{dw}=\frac{-2\sum(y-(xw+b))*x}{N}$$

#### **Pendiente $b$:**

* Desde el punto de vista de la pendiente de $b$, si definimos a $b_{1}=b_{0}+\Delta$ entonces:

$$\lim_{\Delta \rightarrow 0} pendiente_b(b_{0}+\Delta,b_{0})= \lim_{\Delta \rightarrow 0} \frac{MSE(y,h(x,w, b_{0}+\Delta))-MSE(y,h(x,w,b_{0}))}{\Delta} =\frac{dMSE(y,h(x,w,b))}{db}$$

* Esta derivada se puede resolver por regla de la cadena:

$$\frac{dMSE(y,h(x,w,b))}{db}=\frac{dMSE(y,h(x,w,b))}{d(h(x,w,b))}.\frac{d(h(x,w,b))}{db}$$

* La primer derivada ya se resolvió previamente.

* La segunda derivada se resuelve así:

$$\frac{dh(x,w,b)}{db}=\frac{d(xw+b)}{db}=1$$

* Finalmente, resulta que:

$$\frac{dMSE(y,h(x,w,b))}{db}=\frac{dMSE(y,h(x,w,b))}{d(h(x,w,b))}.\frac{d(h(x,w,b))}{db}=\frac{-2\sum(y-(xw+b))*1}{N}$$

***
A continuación graficaremos la función de pérdida y veremos cómo se actualiza el valor del parámetro w en cada iteración:

In [ ]:
from matplotlib import rc
from IPython.core.display import HTML
rc('animation', html='jshtml')
from matplotlib.animation import FuncAnimation

lr = 0.5 #tasa de aprendizaje

fig, ax = plt.subplots()

def gradient_descent(frame):
    global w    
    if frame == 0:
      #inicializo w
      w = tf.constant(-3, shape=(1, 1), dtype=tf.float32) 
        
    #genero un vector ws (eje x)
    ws = np.linspace(-3, 9, 51)
    
    #grafico la loss
    ax.clear()
    ax.plot(ws, exp_loss(y_tf, x_tf, ws))
    
    #punto de partida, coord. (xp, yp)
    xp = w.numpy()[0, 0]
    y_hat = lineal(x_tf, w, 0)
    yp = mse(y_tf, y_hat).numpy()
    
    ax.scatter(xp, yp, c='red')
    
    #Gradient Descent 
    with tf.GradientTape() as g:
        g.watch([w])  #seteo cual es el parámetro que usaremos para derivar
        y_hat = lineal(x_tf, w, 0)
        loss = mse(y_tf, y_hat)
    
    gw = g.gradient(loss, [w])[0]  #calcula el gradiente de la pérdida en función del parámetro w
    
    #dibuja una flechita, cuyo largo está relacionado con la magnitud del gradiente y el learning rate
    ax.arrow(x=w.numpy()[0, 0], y=mse(y_tf, lineal(x_tf, w, 0)).numpy(), dy=0, dx=-gw[0, 0].numpy() * lr, width=.08)
    b_tang = yp - xp * gw[0, 0].numpy()
    y_tang = lambda x: x * gw[0, 0].numpy() + b_tang  
    ax.plot([xp-0.5, xp+0.5], [y_tang(xp-0.5), y_tang(xp+0.5)])
    
    w = w - lr * gw  #actualizo el parámetro w
    pass

ani = FuncAnimation(fig = fig, func = gradient_descent, frames = 100, interval = 100, repeat = True)
HTML(ani.to_jshtml())

**Homework**: Modifique el hiperparámetro lr (learning rate) y vea cómo afecta 
al algoritmo de búsqueda del mínimo de la función de costo.

***
### **Ejericio A-3**: 

Ahora buscaremos que el modelo aprenda ambos parámetros (w y b), por lo cual tendremos una función de pérdida 3D.

In [ ]:
from matplotlib import cm

fig = plt.figure()
ax = fig.add_subplot(projection='3d')

# Generamos los datos
w = np.arange(1, 5, 0.1)
b = np.arange(-1, 1, 0.01)

w, b = np.meshgrid(w, b)
e = np.empty_like(w)

for i in range(w.shape[0]):
    for j in range(w.shape[1]):
      y_hat = lineal(x_tf, tf.constant(w[i, j], shape=(1,1), dtype=tf.float32), b[i, j])
      e[i, j] = mse(y_tf, y_hat)

ax.plot_surface(w, b, e, cmap=cm.coolwarm, linewidth=0, antialiased=False)
ax.set_xlabel('Valores de w')
ax.set_ylabel('Valores de b')
ax.set_zlabel('Función de pérdida')

Inicializamos los parámetros de manera aleatoria empleando una distribución uniforme, calculamos el gradiente respecto a *w* y *b*, y actualizamos dichos parámetros.

In [ ]:
w = tf.random.uniform(shape=(1, 1), minval=-1, maxval=1)
b = tf.random.uniform(shape=[], minval=-1, maxval=1)

#Hiperparámetros
epochs = 60 
lr = 0.1 

losses = []

for i in range(epochs):
    with tf.GradientTape() as g:
        g.watch([w, b])  #tengo 2 parámetros 
        loss = mse(y_tf, lineal(x_tf, w, b))
        losses.append(loss.numpy())
    
    gw, gb = g.gradient(loss, [w, b])  #calcula los gradientes para cada parámetro, w y b pueden ser matrices

    #se actualizan los parámetros
    w = w - lr * gw
    b = b - lr * gb

plt.plot(losses)
plt.xlabel('epochs')
plt.ylabel('pérdida (loss)')

print(f'El w final es {w[0, 0]}')
print(f'El b final es {b}')
print(f'Valor loss min {np.min(losses)}')

`NOTAS`:

Los **hiperparámetros** son parámetros que `no se aprenden` durante el proceso de entrenamiento del modelo, sino que deben ser configurados antes de iniciar dicho entrenamiento. 

Influyen en el rendimiento y comportamiento del modelo y pueden ser ajustados para encontrar la configuración óptima que se adapte mejor a los datos y al problema específico que se está abordando.

algunos ejemplos comunes de hiperparámetros en algoritmos de aprendizaje automático:

**Tasa de aprendizaje (learning rate)**: controla el tamaño de los pasos que se toman durante el proceso de optimización. Determina la rapidez con la que el modelo aprende de los datos. Una tasa de aprendizaje alta puede hacer que el modelo oscile o no converja, mientras que una tasa de aprendizaje baja puede hacer que el modelo converja lentamente o quede atrapado en mínimos locales.

**Número de épocas (epochs)**: número de veces que el modelo pasa por el conjunto completo de entrenamiento durante el proceso de entrenamiento. Un mayor número de épocas puede permitir que el modelo mejore su rendimiento a medida que se ajusta a los datos, pero un número excesivamente alto puede llevar al sobreajuste.

**Tamaño del lote (batch size)**: número de ejemplos de entrenamiento que se utilizan en cada paso de actualización del modelo durante el entrenamiento. Un tamaño de lote pequeño (como en el gradiente descendente estocástico) puede hacer que el entrenamiento sea más rápido pero menos estable, mientras que un tamaño de lote grande (como en el gradiente descendente en mini lotes) puede requerir más memoria pero proporcionar actualizaciones de modelo más estables.

**Regularización**: los métodos de regularización, como la regresión de `Ridge` y la regresión de `Lasso`, tienen hiperparámetros que controlan la fuerza de la regularización aplicada para evitar el sobreajuste. Estos hiperparámetros determinan el equilibrio entre ajustar los datos de entrenamiento y reducir la complejidad del modelo.

Estos son solo algunos ejemplos de hiperparámetros comunes, pero existen muchos más dependiendo del algoritmo y del modelo específico que se esté utilizando. 

La selección y ajuste adecuado de los hiperparámetros es un paso crítico en el proceso de construcción de modelos de aprendizaje automático.